# PREPARE FOR NEXT LEVEL FOR ICP

------------------------------------
PIC SET
------------------------------------

YOLO V8 ROI 기반 영역 제한

PASS THROUGH FILTER

AND 연산 및 ROI 마스크 추출

뎁스맵 영역 제한

메디안 필터링

포인트 클라우드화

ROI 기반 크롭

아웃 라이어 제거

DBSCAN 클러스터링

------------------------------------
COARSE FITTING
------------------------------------

복셀 다운 샘플링

1차 위치 조정
컨벡스헐 + PCA

색상 구분
PCA 장축 추출
3등분, 각구간별 포인트 클라우드들의 RGB 색조 비율에 따라 로컬 x축 180도 회전 여부 지정

2차 위치 조정
fpfh 특징점 추출 + RANSAC 기반 정렬

------------------------------------
FINE FITTING
------------------------------------

KDTREE 인덱싱 구축

Trimmed CAD 거리 제한

point cloud ICP
=icp시 cad를 뎁스에 맞추기


# icp route
초기 ICP 아이디어


------------------------------------
PIC SET
------------------------------------

YOLO V8 ROI 기반 영역 제한

PASS THROUGH FILTER

AND 연산 및 ROI 마스크 추출

뎁스맵 영역 제한

메디안 필터링

포인트 클라우드화

ROI 기반 크롭

아웃 라이어 제거

DBSCAN 클러스터링

------------------------------------
COARSE FITTING
------------------------------------

복셀 다운 샘플링

1차 위치 조정
컨벡스헐 + PCA

색상 구분
PCA 장축 추출
3등분, 각구간별 포인트 클라우드들의 RGB 색조 비율에 따라 로컬 x축 180도 회전 여부 지정

2차 위치 조정
fpfh 특징점 추출 + RANSAC 기반 정렬

------------------------------------
FINE FITTING
------------------------------------

KDTREE 인덱싱 구축

Trimmed CAD 거리 제한

point cloud ICP
=icp시 cad를 뎁스에 맞추기


--------------------------------------------





# YOLO = CVAT TOOL



### 🚀 최종 6D 포즈 추정 파이프라인 (Code-Level Blueprint)

#### 🛡️ [ STEP 1: PIC SET ] (2D-3D 센서 퓨전 및 전처리)

2D 상태에서 지저분한 데이터를 싹 다 날려버려, 3D 연산 부하를 최소화하는 단계입니다.

1. **YOLOv8-Seg 추론:** RGB 이미지에서 듀플로 조립체의 2D 바이너리 마스크 추출
2. **Depth Z축 임계값 마스크:** 바닥(테이블) 높이 이상으로 솟아오른 영역만 1로 마스킹
3. **⭐ 교차 검증 (AND 연산):** `YOLO 마스크 & Depth 마스크`로 허공의 로봇 팔과 바닥의 그림자를 동시 제거
4. **뎁스맵 영역 제한:** 원본 뎁스 이미지에 최종 마스크를 곱해(Element-wise) 배경 데이터를 수학적 `0`으로 소거
5. **메디안 필터링 (2D):** 마스킹된 뎁스맵의 소금-후추 픽셀 노이즈 제거
6. **포인트 클라우드화 (3D 변환):** 카메라 Intrinsics를 적용해 순도 100% 3D 데이터로 변환 (이미 배경이 다 날아갔으므로 PassThrough 필터는 생략 가능!)
7. **아웃라이어 제거:** 허공에 뜬 미세 3D 노이즈 제거
8. **DBSCAN 클러스터링:** 최종적으로 가장 크고 깨끗한 단일 타겟 덩어리 확보 (`Target_Fine`으로 저장)

#### 🧭 [ STEP 2: COARSE FITTING ] (대략적 위치 및 방향 정렬)

무거운 3D 우주 공간에서 CAD 모델을 타겟 근처로 훅! 끌어다 놓는 단계입니다.

1. **복셀 다운 샘플링:** 연산 속도를 위해 타겟을 듬성듬성하게 압축 (`Target_Coarse`)
2. **1차 위치 조정 (PCA + 컨벡스 헐):** 전체적인 덩어리의 중심과 주축(방향)을 대략적으로 매칭
3. **⭐ 색상 기반 대칭 보정:** PCA 장축을 3등분하여 RGB 비율 확인 $\rightarrow$ 역순일 경우 로컬 $X$축 기준 180도 회전 적용 (위아래 완벽 정렬)
4. **안전한 이동:** CAD 모델이 뎁스 껍데기 아래로 푹 꺼지지 않도록 Z축 최상단(또는 바운딩 박스) 기준으로 끌어올림
5. **2차 위치 조정 (FPFH + RANSAC):** 듀플로 특유의 모서리/굴곡 지문을 추출해 밀리미터 단위로 초기 뼈대 강제 결합

#### 🎯 [ STEP 3: FINE FITTING ] (0.1mm 단위 초정밀 밀착)

어긋난 표면을 톱니바퀴처럼 완벽하게 맞물려 잠그는(Lock-in) 단계입니다.

1. **고해상도 타겟 로드:** 아까 STEP 1에서 저장해둔 빽빽한 `Target_Fine`과 CAD 모델을 준비
2. **KD-Tree 인덱싱 자동 구축:** 초고속 이웃 탐색을 위한 공간 분할
3. **⭐ 거리 제한 설정 (Trimmed):** 반쪽짜리 카메라 뷰의 한계를 극복하기 위해, 안 보이는 CAD 뒷면 점들을 매칭에서 강제 제외
4. **Point-to-Plane ICP 진행:** CAD의 면(Plane)을 뎁스 표면에 완벽하게 밀착 슬라이딩시켜 최종 $4 \times 4$ 변환 행렬($T$) 도출

#### 🤖 [ STEP 4: OUTPUT ] (로봇 제어용 출력)

1. **90도 스냅 (Snap) 보정:** $T$ 행렬의 회전각을 로봇이 잡기 가장 편한 90도 정방향 단위로 반올림하여 미세 회전 유격 제거
2. **ROS2 TF 브로드캐스팅:** 최종 오일러 각(Roll, Pitch, Yaw)과 XYZ 좌표를 로봇 팔(매니퓰레이터) 제어 노드로 전송

---